# Agente Funcional ChileAtiende — EP2 ISY0101
**Evaluación Parcial N°2 — Ingeniería de Soluciones con IA**

Notebook interactivo para Google Colab. Ejecuta las celdas en orden (1 a 6).

> **Requisito:** Necesitas una API Key de OpenAI. Pégala en la Celda 2.

## Celda 1 — Instalar dependencias

In [ ]:
# Instalar todas las librerias necesarias (incluye langchain-text-splitters)
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters chromadb tiktoken openai
print('\n--- Dependencias instaladas correctamente ---')

## Celda 2 — Configurar API Key
Reemplaza el texto entre comillas con tu clave real de OpenAI.

In [ ]:
import os

# ============================
# PEGA TU API KEY AQUI ABAJO:
# ============================
os.environ['OPENAI_API_KEY'] = 'TU_API_KEY_AQUI'

# Verificacion rapida
key = os.environ.get('OPENAI_API_KEY', '')
if key == 'TU_API_KEY_AQUI' or len(key) < 10:
    print('ADVERTENCIA: No has pegado tu API Key. Reemplaza TU_API_KEY_AQUI con tu clave real.')
else:
    print(f'API Key configurada correctamente (termina en ...{key[-4:]})')

## Celda 3 — Crear Base de Conocimientos
Se crea el archivo con fichas de tramites de ChileAtiende directamente en Colab.

In [ ]:
import os
os.makedirs('data', exist_ok=True)

fichas = '''=== PGU (Pension Garantizada Universal) ===
Descripcion: Beneficio del Estado que reemplaza los beneficios del Pilar Solidario, pagado por el IPS.
Requisitos:
- Tener 65 anos o mas de edad.
- No integrar el 10 por ciento mas rico de la poblacion.
- Acreditar residencia en Chile por 20 anos continuos o discontinuos, o 4 de los ultimos 5 anos previos a la solicitud.
- Contar con una pension base menor a 1.158.355 pesos.
Monto: El monto maximo es de 214.296 pesos mensuales.
Fuente: https://www.chileatiende.gob.cl/fichas/65066-pension-garantizada-universal-pgu

=== Bono al Trabajo de la Mujer (BTM) ===
Descripcion: Beneficio en dinero para mejorar los ingresos de las mujeres trabajadoras.
Requisitos:
- Tener entre 25 y 59 anos de edad.
- Pertenecer al 40 por ciento mas vulnerable segun el Registro Social de Hogares (RSH).
- Mantener las cotizaciones al dia.
- No trabajar en una Institucion del Estado.
Monto: Depende de la renta y puede pagarse de forma anual o mensual.
Fuente: https://www.chileatiende.gob.cl/fichas/1285-bono-al-trabajo-de-la-mujer

=== Subsidio de Arriendo ===
Descripcion: Beneficio habitacional del MINVU para arrendar una vivienda.
Requisitos:
- Ser mayor de 18 anos.
- No ser propietario de una vivienda.
- Pertenecer al 40 por ciento mas vulnerable segun el RSH.
- No haber sido beneficiado previamente con un subsidio habitacional.
Monto: Varia segun zona geografica y puede llegar hasta 5 UF mensuales.
Fuente: https://www.chileatiende.gob.cl/fichas/11281-subsidio-de-arriendo

=== Bono de Invierno ===
Descripcion: Beneficio anual pagado entre mayo y agosto para gastos de calefaccion.
Requisitos:
- Tener 65 anos o mas al 31 de mayo del ano del pago.
- Ser beneficiario de PGU, Pension Basica Solidaria o Aporte Previsional Solidario.
Monto: Varia cada ano, historicamente entre 50.000 y 100.000 pesos en un solo pago.
Fuente: https://www.chileatiende.gob.cl/fichas/2213-bono-de-invierno

=== Asignacion Familiar ===
Descripcion: Beneficio economico mensual para trabajadores y pensionados por cada carga familiar.
Requisitos:
- Ser trabajador dependiente, pensionado o imponente de alguna Caja de Prevision.
- Acreditar cargas familiares (hijos hasta 18 anos, o hasta 24 si son estudiantes).
Monto: Depende del tramo de renta del causante.
Fuente: https://www.chileatiende.gob.cl/fichas/14901-asignacion-familiar'''

with open('data/fichas_tramites.txt', 'w', encoding='utf-8') as f:
    f.write(fichas)

print(f'Base de conocimientos creada: {len(fichas)} caracteres, 5 fichas de tramites.')

## Celda 4 — Cargar datos + Definir Herramientas del Agente
Se construye el vectorstore RAG y se definen las 3 herramientas que el agente puede usar.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.tools import tool

# --- Paso 1: Cargar y fragmentar documentos ---
print('Cargando documentos...')
loader = TextLoader('data/fichas_tramites.txt', encoding='utf-8')
docs = loader.load()

splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separator='\n\n'
)
splits = splitter.split_documents(docs)
print(f'{len(splits)} fragmentos generados.')

# --- Paso 2: Crear vectorstore con embeddings ---
print('Generando embeddings y creando base vectorial...')
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
print('Base vectorial creada.')

# --- Paso 3: Definir las 3 herramientas del agente ---

@tool
def buscar_tramite(consulta: str) -> str:
    """Busca informacion oficial sobre bonos, beneficios y tramites del Estado de Chile
    en la base de conocimientos de ChileAtiende. Usala cuando el ciudadano pregunte
    sobre requisitos, montos o como postular a un beneficio estatal."""
    resultados = vectorstore.similarity_search(consulta, k=2)
    if not resultados:
        return 'No se encontro informacion relevante en la base de datos.'
    partes = []
    for doc in resultados:
        partes.append(doc.page_content)
    return '\n\n---\n\n'.join(partes)

@tool
def redactar_resumen(informacion: str) -> str:
    """Toma informacion sobre un tramite o beneficio y redacta un resumen claro,
    formal y empatico para el ciudadano. Usala despues de buscar_tramite
    para formatear la respuesta final. Maximo 5 oraciones."""
    llm_resumen = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.2)
    mensaje = (
        'Eres un redactor oficial de ChileAtiende. '
        'Transforma la siguiente informacion en un resumen claro, empatico y formal '
        'para un ciudadano. Usa lenguaje inclusivo. Maximo 5 oraciones.\n\n'
        'Informacion:\n' + informacion
    )
    resp = llm_resumen.invoke(mensaje)
    return resp.content

@tool
def derivar_ejecutivo(motivo: str) -> str:
    """Genera un mensaje de derivacion a un ejecutivo humano de ChileAtiende.
    Usala cuando la consulta involucra datos personales (RUT, estado de solicitud),
    cuando la informacion no esta en la base de conocimientos, o cuando el
    ciudadano necesita atencion personalizada."""
    return (
        'Entiendo que su consulta requiere atencion personalizada (' + motivo + '). '
        'Le recomendamos contactar directamente con un ejecutivo de ChileAtiende:\n'
        'Call Center: 101 (lunes a viernes, 08:00 a 22:00 hrs)\n'
        'Web: https://www.chileatiende.gob.cl\n'
        'Sucursales: https://www.chileatiende.gob.cl/sucursales\n'
        'Un ejecutivo podra revisar su caso con total confidencialidad.'
    )

TOOLS = [buscar_tramite, redactar_resumen, derivar_ejecutivo]
print(f'\nHerramientas registradas: {[t.name for t in TOOLS]}')

## Celda 5 — Construir el Agente con Memoria
Se crea el AgentExecutor con el patrón **ReAct** (Reasoning + Acting), memoria de corto plazo y las 3 herramientas.

In [ ]:
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# --- System Prompt del Agente ---
SYSTEM_PROMPT = (
    'Eres un agente inteligente oficial de ChileAtiende, '
    'el sistema de atencion ciudadana del Estado de Chile.\n\n'
    'Tus responsabilidades son:\n'
    '1. Responder consultas sobre bonos, beneficios y tramites '
    'usando la herramienta buscar_tramite.\n'
    '2. Formatear las respuestas de forma clara usando redactar_resumen.\n'
    '3. Derivar al ciudadano con derivar_ejecutivo cuando la consulta '
    'involucra datos personales (RUT), o no se encuentra en la base.\n\n'
    'Reglas estrictas:\n'
    '- NUNCA inventes requisitos o montos.\n'
    '- Usa tono formal, empatico e inclusivo.\n'
    '- Recuerda el contexto previo de la conversacion.\n\n'
    'Historial: {chat_history}'
)

# --- Configurar LLM, Prompt, Agente y Memoria ---
llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{input}'),
    MessagesPlaceholder(variable_name='agent_scratchpad'),
])

agent = create_openai_functions_agent(llm=llm, tools=TOOLS, prompt=prompt)

# Memoria de corto plazo: ultimos 5 turnos
memory = ConversationBufferWindowMemory(
    k=5,
    memory_key='chat_history',
    return_messages=True,
    input_key='input',
)

# Ejecutor del agente
agent_executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,
)

print('Agente listo! Puedes ejecutar la celda siguiente para probarlo.')

## Celda 6 — Demostracion: 3 Consultas de Prueba
Se ejecutan 3 consultas que demuestran:
1. **Busqueda RAG** con respuesta formateada
2. **Memoria de corto plazo** (continuidad sin repetir contexto)
3. **Derivacion a ejecutivo** (datos personales / fuera de scope)

In [ ]:
print('='*60)
print('  CONSULTA 1: Requisitos de un beneficio')
print('='*60)
r1 = agent_executor.invoke(
    {'input': 'Hola, tengo 67 anos. Cuales son los requisitos para la PGU?'}
)
print('\nRESPUESTA FINAL:')
print(r1['output'])

print('\n' + '='*60)
print('  CONSULTA 2: Prueba de memoria (sin repetir contexto)')
print('='*60)
r2 = agent_executor.invoke(
    {'input': 'Y cuanto dinero me darian cada mes?'}
)
print('\nRESPUESTA FINAL (memoria):')
print(r2['output'])

print('\n' + '='*60)
print('  CONSULTA 3: Derivacion a ejecutivo humano')
print('='*60)
r3 = agent_executor.invoke(
    {'input': 'Necesito saber el estado de mi solicitud, mi RUT es 12.345.678-9'}
)
print('\nRESPUESTA FINAL (derivacion):')
print(r3['output'])

## Celda 7 (Opcional) — Chat interactivo
Puedes conversar libremente con el agente. Escribe `salir` para terminar.

In [ ]:
print('Chat interactivo con el Agente ChileAtiende')
print('Escribe "salir" para terminar.\n')

while True:
    pregunta = input('Ciudadano: ')
    if pregunta.strip().lower() in ('salir', 'exit', 'quit'):
        print('Agente: Hasta pronto! Que tenga un excelente dia.')
        break
    try:
        resp = agent_executor.invoke({'input': pregunta})
        print(f"\nAsistente ChileAtiende: {resp['output']}\n")
    except Exception as e:
        print(f'Error: {e}\n')